# MA429 Summative Project (Section 9)

- **Section 9 (Unified Comparison)**: This notebook provides a unified comparison of forecasting results from `Section 5` to `Section 8`, and adds harmonized baseline methods:

- `Naive` (previous time step)
- `Seasonal Naive` (same hour in the previous week)


---
## 9 Unified Comparison
### 9.1 Imports

In [6]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd()

ROW_ORDER = [
    "Naive",
    "Seasonal Naive",
    "XGboost (lag and rolling)",
    "Random Forest (lag and rolling)",
    "XGboost (no lag and no rolling)",
    "Random Forest (no lag and no rolling)",
    "LSTM",
    "GAT + LSTM",
    "GAT + LSTM (improved)",
]

METRIC_COLS = ["MAE", "RMSE", "WAPE", "R2"]

TARGET_MAP = {
    "target_outflow_t1": "outflow",
    "target_inflow_t1": "inflow",
}

PANEL_CANDIDATES = [
    PROJECT_ROOT / "station_hour_features.parquet",
    PROJECT_ROOT / "station_hour_panel.parquet",
    PROJECT_ROOT / "data" / "station_hour_features.parquet",
    PROJECT_ROOT / "data" / "station_hour_panel.parquet",
    PROJECT_ROOT / "station-hour panel.parquet",
    PROJECT_ROOT / "data" / "station-hour panel.parquet",
]

SECTION_RESULT_CANDIDATES = {
    "section5": [
        PROJECT_ROOT / "section5_results" / "baseline_results.csv",
        PROJECT_ROOT / "results" / "section5_results" / "baseline_results.csv",
    ],
    "section6": [
        PROJECT_ROOT / "section6_results" / "baseline_results.csv",
        PROJECT_ROOT / "results" / "section6_results" / "baseline_results.csv",
    ],
    "section7": [
        PROJECT_ROOT / "section7_results" / "gat_lstm_results.csv",
        PROJECT_ROOT / "results" / "section7_results" / "gat_lstm_results.csv",
    ],
    "section8": [
        PROJECT_ROOT / "section8 results" / "gat_lstm_results.csv",
        PROJECT_ROOT / "section7_results_two stage" / "gat_lstm_results.csv",
        PROJECT_ROOT / "results" / "section8 results" / "gat_lstm_results.csv",
        PROJECT_ROOT / "results" / "section7_results_two stage" / "gat_lstm_results.csv",
    ],
}


## 9.2 Utility Functions


In [7]:
def resolve_first_existing(candidates: List[Path], label: str) -> Path:
    for p in candidates:
        if p.exists():
            return p
    candidate_text = "\n".join(f"- {c}" for c in candidates)
    raise FileNotFoundError(f"{label} not found. Tried:\n{candidate_text}")

def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def wape(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-12) -> float:
    denom = float(np.sum(np.abs(y_true)))
    if denom < eps:
        return np.nan
    return float(np.sum(np.abs(y_true - y_pred)) / denom)

def r2(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-12) -> float:
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    y_bar = float(np.mean(y_true))
    ss_tot = float(np.sum((y_true - y_bar) ** 2))
    if ss_tot < eps:
        return np.nan
    return float(1.0 - ss_res / ss_tot)

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "WAPE": wape(y_true, y_pred),
        "R2": r2(y_true, y_pred),
    }

def temporal_splits(n: int, train_ratio: float = 0.70, val_ratio: float = 0.15) -> Tuple[slice, slice, slice]:
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    return slice(0, n_train), slice(n_train, n_train + n_val), slice(n_train + n_val, n)

def build_baseline_metrics(df_panel: pd.DataFrame) -> Dict[str, Dict[str, float]]:
    df = df_panel.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["station_id", "date", "hour"]).reset_index(drop=True)
    results = {}
    for target_col, target_name in TARGET_MAP.items():
        if target_col not in df.columns:
            raise KeyError(f"Missing target column in panel: {target_col}")
        y = df[target_col]
        naive_pred = y.shift(1)
        seasonal_pred = y.shift(24 * 7)
        valid = pd.DataFrame({"y": y, "naive": naive_pred, "seasonal": seasonal_pred}).dropna()
        _, _, test_slice = temporal_splits(len(valid))
        test_df = valid.iloc[test_slice].copy()
        y_true = test_df["y"].to_numpy()
        y_naive = test_df["naive"].to_numpy()
        y_seasonal = test_df["seasonal"].to_numpy()
        results[target_name] = {
            "naive": compute_metrics(y_true, y_naive),
            "seasonal naive": compute_metrics(y_true, y_seasonal),
        }
    return results

def pick_metrics(df: pd.DataFrame, model: str, target: str, split: str = "test", feature_set: str | None = None) -> Dict[str, float]:
    q = df[(df["model"] == model) & (df["target"] == target) & (df["split"] == split)]
    if feature_set is not None:
        q = q[q["feature_set"] == feature_set]
    if q.empty:
        raise ValueError(f"No row found for model={model}, target={target}, split={split}, feature_set={feature_set}")
    row = q.iloc[0]
    return {m: float(row[m]) for m in METRIC_COLS}


## 9.3 Load Station-Hour Panel and Section 5-8 Result Files


In [8]:
panel_path = resolve_first_existing(PANEL_CANDIDATES, "station-hour panel parquet")
section5_path = resolve_first_existing(SECTION_RESULT_CANDIDATES["section5"], "Section 5 results")
section6_path = resolve_first_existing(SECTION_RESULT_CANDIDATES["section6"], "Section 6 results")
section7_path = resolve_first_existing(SECTION_RESULT_CANDIDATES["section7"], "Section 7 results")
section8_path = resolve_first_existing(SECTION_RESULT_CANDIDATES["section8"], "Section 8 results")

print("Panel:", panel_path)
print("Section 5:", section5_path)
print("Section 6:", section6_path)
print("Section 7:", section7_path)
print("Section 8:", section8_path)

df_panel = pd.read_parquet(panel_path)
res5 = pd.read_csv(section5_path)
res6 = pd.read_csv(section6_path)
res7 = pd.read_csv(section7_path)
res8 = pd.read_csv(section8_path)

print("\nLoaded shapes:")
print("panel:", df_panel.shape)
print("section5:", res5.shape)
print("section6:", res6.shape)
print("section7:", res7.shape)
print("section8:", res8.shape)


Panel: /Users/gurunchi/Desktop/MA429 Summative Project Coding/station_hour_features.parquet
Section 5: /Users/gurunchi/Desktop/MA429 Summative Project Coding/section5_results/baseline_results.csv
Section 6: /Users/gurunchi/Desktop/MA429 Summative Project Coding/section6_results/baseline_results.csv
Section 7: /Users/gurunchi/Desktop/MA429 Summative Project Coding/section7_results/gat_lstm_results.csv
Section 8: /Users/gurunchi/Desktop/MA429 Summative Project Coding/section8 results/gat_lstm_results.csv

Loaded shapes:
panel: (21132864, 53)
section5: (16, 8)
section6: (4, 8)
section7: (4, 8)
section8: (4, 8)


## 9.4 Build Unified Metric Tables for Outflow and Inflow

Combine standardized baselines and Section 5-8 metrics, then generate two final tables using the required row order.


In [9]:
baseline_metrics = build_baseline_metrics(df_panel)

def build_table_for_target(target_col: str, target_name: str) -> pd.DataFrame:
    table_rows = {
        "Naive": baseline_metrics[target_name]["naive"],
        "Seasonal Naive": baseline_metrics[target_name]["seasonal naive"],
        "XGboost (lag and rolling)": pick_metrics(res5, model="XGBoost", target=target_col, split="test", feature_set="all_het_lag_rolling"),
        "Random Forest (lag and rolling)": pick_metrics(res5, model="RandomForest", target=target_col, split="test", feature_set="all_het_lag_rolling"),
        "XGboost (no lag and no rolling)": pick_metrics(res5, model="XGBoost", target=target_col, split="test", feature_set="all_het_no_lag_no_roll"),
        "Random Forest (no lag and no rolling)": pick_metrics(res5, model="RandomForest", target=target_col, split="test", feature_set="all_het_no_lag_no_roll"),
        "LSTM": pick_metrics(res6, model="LSTM", target=target_col, split="test", feature_set="base_all_heterogeneity"),
        "GAT + LSTM": pick_metrics(res7, model="GAT_LSTM", target=target_col, split="test", feature_set="base_all_heterogeneity"),
        "GAT + LSTM (improved)": pick_metrics(res8, model="GAT_LSTM", target=target_col, split="test", feature_set="base_all_heterogeneity"),
    }
    out = pd.DataFrame.from_dict(table_rows, orient="index")
    out = out.loc[ROW_ORDER, METRIC_COLS]
    out.index.name = "model"
    return out

outflow_table = build_table_for_target("target_outflow_t1", "outflow").round(6)
inflow_table = build_table_for_target("target_inflow_t1", "inflow").round(6)


## 9.5 Display and Export Final Tables

Display the final comparison tables and export them as CSV files for direct use in the report.


In [10]:
print("Outflow comparison table")
display(outflow_table)

print("Inflow comparison table")
display(inflow_table)

output_dir = PROJECT_ROOT / "section9_results"
output_dir.mkdir(parents=True, exist_ok=True)

outflow_path = output_dir / "outflow_model_comparison.csv"
inflow_path = output_dir / "inflow_model_comparison.csv"

outflow_table.to_csv(outflow_path)
inflow_table.to_csv(inflow_path)

print("\nSaved:")
print(outflow_path)
print(inflow_path)


Outflow comparison table


,MAE,RMSE,WAPE,R2
model,,,,
Naive,0.717030,1.608816,0.935314,0.272387
Seasonal Naive,0.709048,1.616882,0.924902,0.265073
XGboost (lag and rolling),0.201646,0.702546,0.747590,0.648274
Random Forest (lag and rolling),0.196513,0.710483,0.728562,0.640283
XGboost (no lag and no rolling),0.206737,0.753552,0.766465,0.595349
Random Forest (no lag and no rolling),0.199106,0.744548,0.738172,0.604962
LSTM,0.218805,0.780124,0.813559,0.564592
GAT + LSTM,0.219460,0.720881,0.813507,0.629580
GAT + LSTM (improved),0.201897,0.722802,0.748401,0.627603


Inflow comparison table


,MAE,RMSE,WAPE,R2
model,,,,
Naive,0.707472,1.596705,0.920076,0.311413
Seasonal Naive,0.701572,1.610489,0.912404,0.299473
XGboost (lag and rolling),0.204270,0.737136,0.757288,0.615667
Random Forest (lag and rolling),0.199929,0.750657,0.741194,0.601438
XGboost (no lag and no rolling),0.208994,0.780121,0.774799,0.569536
Random Forest (no lag and no rolling),0.201679,0.771795,0.747681,0.578675
LSTM,0.272288,0.811909,1.012419,0.532358
GAT + LSTM,0.218614,0.733216,0.810371,0.619671
GAT + LSTM (improved),0.202692,0.743816,0.751350,0.608595



Saved:
/Users/gurunchi/Desktop/MA429 Summative Project Coding/section9_results/outflow_model_comparison.csv
/Users/gurunchi/Desktop/MA429 Summative Project Coding/section9_results/inflow_model_comparison.csv
